# FX Forecast Report

## Five-Day FX Forecast for Any Currency Pair

A repeatable, parameterized notebook that produces a **five-day-horizon FX forecast** for any currency pair — a directional view, the key drivers, and the key risks. It combines **Bigdata.com** data pulls (country tearsheets + news/research search) with an **OpenAI** synthesis layer.

> All Bigdata.com data is retrieved via the **remote MCP server** at `https://mcp.bigdata.com/` (streamable HTTP, authenticated with an API key).

### Client-level workflow

![FX Forecast Report workflow](assets/fx-forecast-workflow.svg)

[Open the full-size workflow diagram](assets/fx-forecast-workflow.svg)

**Client talk track:** define the pair once; Bigdata.com MCP gathers live structured and qualitative evidence; six explainable FX drivers are scored with configurable weights; the output is a five-day directional call with conviction, risks, and auditable sources.

### Workflow

1. **Parameters** — set base/quote country, pair, horizon, central-bank names, optional sector export terms, an intervention-history flag, and (optionally) weight overrides.
2. **Data layer** — pull the base and quote `bigdata_country_tearsheet`s (skipping unsupported countries) and parse the economic calendar for events inside the horizon.
3. **Central-bank feed** — generate a monetary-policy lexicon per central bank with the LLM and search in parallel; this evidence feeds the rate-differential driver.
4. **Driver evidence** — run the parameterized `bigdata_search` set for the remaining drivers.
5. **Scoring & aggregation** — the LLM scores each driver (lean / confidence / rationale / sources); a weighted blend yields the overall call.
6. **Report** — executive summary, driver table, risk flags, and a source appendix, rendered here and saved to `output/`.

### Coverage note

`bigdata_country_tearsheet` supports a fixed set of 42 countries and has no currency-pair argument. **Taiwan is not supported**, so `USD/TWD` runs with a structured US (base) side and a **search-only** Taiwan (quote) side. The worked example below uses **`USD/JPY`**, where both sides are fully supported and Japan's MoF/BoJ intervention history exercises the intervention-risk driver.

## Step 1 — Environment setup

Set up the environment with `uv` and provide credentials.

```bash
# From the FX_Forecast_Report/ directory
uv venv
source .venv/bin/activate        # Windows: .venv\Scripts\activate
uv pip install -r requirements.txt
uv run jupyter lab
```

Then export `BIGDATA_API_KEY` and `OPENAI_API_KEY`, or copy `.env.example` to `.env` and fill them in. Requires Python 3.11+.

In [1]:
import os
import sys
from pathlib import Path

import nest_asyncio
import pandas as pd
from dotenv import load_dotenv
from IPython.display import Markdown, display

# Jupyter runs inside an event loop; nest_asyncio lets us use top-level `await`.
nest_asyncio.apply()

# Make the project packages importable.
NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

load_dotenv()

bigdata_key = os.getenv("BIGDATA_API_KEY")
openai_key = os.getenv("OPENAI_API_KEY")
print(f"Bigdata API Key: {'Set' if bigdata_key else 'MISSING'}")
print(f"OpenAI API Key:  {'Set' if openai_key else 'MISSING'}")
if not bigdata_key or not openai_key:
    print("\nSet BIGDATA_API_KEY and OPENAI_API_KEY in your shell or a .env file.")

Bigdata API Key: Set
OpenAI API Key:  Set


In [2]:
from src.bigdata_mcp_client import BigdataMCPClient
from src.central_bank_feed import gather_central_bank_evidence
from src.data_layer import ForecastParams, build_driver_queries, pull_tearsheets
from src.llm import LLMClient
from src.report import assemble_report, save_report
from src.scoring import aggregate, group_evidence_by_label, score_all_drivers

client = BigdataMCPClient(api_key=bigdata_key)
llm = LLMClient()
print(f"Bigdata MCP client + OpenAI ({llm.model}) initialized.")

Bigdata MCP client + OpenAI (gpt-4o-mini) initialized.


## Step 2 — Configure the forecast

Everything below the parameters is pair-agnostic. To forecast a different pair, change only this cell.

- `USD/JPY` (default): both countries fully supported; `intervention_history=True` activates the intervention driver.
- `USD/TWD` (generalization target): set `quote_country="TW"` — the quote side falls back to search only (see the commented block).

In [3]:
params = ForecastParams(
    base_country="US",
    quote_country="JP",
    pair="USD/JPY",
    central_bank_base="Federal Reserve",
    central_bank_quote="Bank of Japan",
    horizon_days=5,
    sector_driver_terms=["auto exports", "semiconductor exports"],
    intervention_history=True,
)

# --- Generalization example: USD/TWD (Taiwan quote side is search-only) ---
# params = ForecastParams(
#     base_country="US",
#     quote_country="TW",
#     pair="USD/TWD",
#     central_bank_base="Federal Reserve",
#     central_bank_quote="Central Bank of the Republic of China (Taiwan)",
#     horizon_days=5,
#     sector_driver_terms=["semiconductor exports", "TSMC"],
#     intervention_history=True,
# )

print(f"Pair: {params.pair}  (base {params.base_ccy} / quote {params.quote_ccy})")
print(f"Horizon: {params.horizon_days} days")
print("Normalized driver weights:")
for key, weight in params.weights().items():
    print(f"  {key:24s} {weight:.0%}")

Pair: USD/JPY  (base USD / quote JPY)
Horizon: 5 days
Normalized driver weights:
  rate_differential        35%
  trade_capital_flows      10%
  intervention_risk        20%
  risk_sentiment_carry     20%
  geopolitical             5%
  technical_positioning    10%


## Step 3 — Data layer: country tearsheets + economic calendar

Pull the base and quote tearsheets in one shot each (FX pricing, economic calendar, sectoral macro, market indices, sovereign yields, G7 comparison) and extract the calendar events landing inside the horizon. Unsupported countries return `None` and are noted.

In [4]:
bundle = await pull_tearsheets(client, params)

print(f"Base tearsheet ({params.base_name}):  {'available' if bundle.base_available else 'UNAVAILABLE'}")
print(f"Quote tearsheet ({params.quote_name}): {'available' if bundle.quote_available else 'UNAVAILABLE'}")
for note in bundle.notes:
    print(f"  note: {note}")
print(f"\nEconomic events inside the {params.horizon_days}-day horizon: {len(bundle.events)}")

events_df = pd.DataFrame(
    [
        {
            "Date (UTC)": e.date.strftime("%Y-%m-%d %H:%M"),
            "Country": e.country,
            "Impact": e.impact.upper(),
            "Event": e.event,
            "Consensus": e.consensus,
        }
        for e in bundle.events
    ]
)
events_df

Base tearsheet (United States):  available
Quote tearsheet (Japan): available

Economic events inside the 5-day horizon: 29


,Date (UTC),Country,Impact,Event,Consensus
0,2026-07-24 17:00,US,LOW,Baker Hughes US Oil Rig Count,
1,2026-07-24 19:30,US,LOW,CFTC Gold NC Net Positions,
2,2026-07-24 19:30,US,LOW,CFTC Oil NC Net Positions,
3,2026-07-24 19:30,US,LOW,CFTC S&P 500 NC Net Positions,
4,2026-07-24 19:30,JP,LOW,CFTC JPY NC Net Positions,
5,2026-07-26 23:50,JP,LOW,Corporate Service Price Index (YoY) (Jun),
6,2026-07-27 05:00,JP,LOW,Coincident Index (May),
7,2026-07-27 05:00,JP,LOW,Leading Economic Index (May),116.8
8,2026-07-27 12:30,US,MEDIUM,Durable Goods Orders (Jun),1.6%
9,2026-07-27 12:30,US,MEDIUM,Durable Goods Orders ex Defense (Jun),


## Step 4 — Central-bank sentiment feed

Generate a monetary-policy lexicon for each central bank (dynamically, not hardcoded), search in parallel, and collect the evidence that feeds the **rate-differential** driver.

In [5]:
cb_feed = await gather_central_bank_evidence(client, llm, params, per_bank=5, max_docs=16)

print(f"{params.central_bank_base} lexicon:")
for term in cb_feed.base_terms:
    print(f"  - {term}")
print(f"\n{params.central_bank_quote} lexicon:")
for term in cb_feed.quote_terms:
    print(f"  - {term}")
print(f"\nCentral-bank evidence documents: {len(cb_feed.evidence)}")

Federal Reserve lexicon:
  - What is the latest Federal Reserve policy-rate decision and near-term rate expectations for the USD?
  - How is the Federal Reserve's forward guidance characterized in terms of hawkish or dovish tone?
  - What is the current inflation trajectory in relation to the Federal Reserve's target?
  - When is the next Federal Reserve policy meeting and what have officials recently commented on monetary policy?
  - What recent statements from the Federal Reserve indicate their stance on interest rates and economic outlook?

Bank of Japan lexicon:
  - What is the latest policy-rate decision from the Bank of Japan and what are the near-term rate expectations for the JPY?
  - How is the Bank of Japan's forward guidance characterized in terms of hawkish or dovish tone?
  - What is the current inflation trajectory in Japan compared to the Bank of Japan's target?
  - When is the next policy meeting of the Bank of Japan and what have officials recently commented on monetar

## Step 5 — Driver evidence search

Run the parameterized `bigdata_search` set for the remaining drivers (trade & capital flows, intervention risk, risk sentiment / carry, geopolitical, technical / positioning), then attach the central-bank feed to the rate-differential driver.

In [6]:
driver_specs = build_driver_queries(params)
grouped = await client.search_many(driver_specs)
evidence_by_driver = group_evidence_by_label(grouped)
evidence_by_driver["rate_differential"] = cb_feed.evidence

print("Evidence documents per driver:")
for key, docs in evidence_by_driver.items():
    print(f"  {key:24s} {len(docs)}")
if client.errors:
    print("\nNon-fatal search errors:", client.errors)

Evidence documents per driver:
  trade_capital_flows      29
  intervention_risk        11
  risk_sentiment_carry     19
  geopolitical             22
  technical_positioning    12
  rate_differential        16


## Step 6 — Driver scoring & aggregation

The LLM scores each active driver (directional lean, confidence, one-line rationale, sources) from the tearsheet context + retrieved evidence, then a weighted blend produces the overall directional call for the horizon.

In [7]:
results = score_all_drivers(
    llm, params, bundle.base_markdown, bundle.quote_markdown, evidence_by_driver
)
agg = aggregate(params, results)

scores_df = pd.DataFrame(
    [
        {
            "Driver": r.label,
            "Lean": r.lean,
            "Confidence": round(r.confidence, 2),
            "Weight": f"{r.weight:.0%}",
            "Contribution": round(r.contribution, 3),
            "Sources": len(r.sources),
            "Rationale": r.rationale,
        }
        for r in results
    ]
)
display(scores_df)

print(f"\nOVERALL CALL: {agg.direction_text}")
print(f"Conviction: {agg.conviction_label} (net score {agg.net_score:+.2f})")

,Driver,Lean,Confidence,Weight,Contribution,Sources,Rationale
0,Rate Differential,base_up,0.8,35%,0.280,3,The Fed is expected to maintain its interest r...
1,Trade & Capital Flows,base_down,0.7,10%,-0.070,5,"Japan's strong export performance, particularl..."
2,Intervention Risk,base_up,0.7,20%,0.140,5,"Despite intervention risks, the persistent wea..."
3,Risk Sentiment / Carry,base_up,0.8,20%,0.160,3,The combination of rising US interest rate exp...
4,Geopolitical,base_up,0.7,5%,0.035,3,Geopolitical tensions and a resilient US econo...
5,Technical / Positioning,base_up,0.8,10%,0.080,5,The technical indicators suggest a bullish tre...



OVERALL CALL: USD appreciates vs JPY (USD/JPY rises)
Conviction: High (net score +0.62)


## Step 7 — Assemble the report

Render the full markdown report (executive summary, driver table, risk flags, source appendix) and save a copy to `output/`.

In [8]:
report_md = assemble_report(params, bundle, cb_feed, results, agg, llm)
report_path = save_report(report_md, params, "output")
print(f"Saved report to: {report_path}\n")
display(Markdown(report_md))

Saved report to: output/USDJPY_2026-07-24_fx_forecast.md



# USD/JPY — 5-Day FX Forecast

*Generated 2026-07-24 16:24 UTC · Base: United States (Federal Reserve) · Quote: Japan (Bank of Japan)* Sector focus: auto exports, semiconductor exports.

## Executive Summary

In the upcoming five days, we anticipate the USD to appreciate against the JPY, resulting in a rise in the USD/JPY pair, with a high conviction level of +0.62. The primary drivers behind this outlook include a favorable rate differential as the Fed maintains its interest rate while the Bank of Japan remains steady, alongside strong risk sentiment bolstered by rising US interest rate expectations and geopolitical tensions. Additionally, the persistent weakness of the JPY, despite Japan's robust export performance, supports our bullish stance. However, intervention risk from Japanese authorities remains a key concern that could impact the trajectory of the USD/JPY pair.

**Overall call:** USD appreciates vs JPY (USD/JPY rises)
**Conviction:** High (net score +0.62)

## Driver Table

| Driver | Lean | Confidence | Weight | Rationale | Sources |
|---|---|---|---|---|---|
| Rate Differential | ▲ base up (USD/JPY ↑) | 0.80 | 35% | The Fed is expected to maintain its interest rate while the Bank of Japan holds steady, creating a favorable rate differential for the USD against the JPY. | 3 doc(s) |
| Trade & Capital Flows | ▼ base down (USD/JPY ↓) | 0.70 | 10% | Japan's strong export performance, particularly in semiconductors and electronics, combined with potential capital inflows from domestic pension funds, supports the JPY against the USD. | 5 doc(s) |
| Intervention Risk | ▲ base up (USD/JPY ↑) | 0.70 | 20% | Despite intervention risks, the persistent weakness of the JPY and the supportive US economic data suggest that the USD will appreciate against the JPY in the near term. | 5 doc(s) |
| Risk Sentiment / Carry | ▲ base up (USD/JPY ↑) | 0.80 | 20% | The combination of rising US interest rate expectations and geopolitical tensions is strengthening the USD while putting downward pressure on the JPY, leading to a bullish outlook for USD/JPY. | 3 doc(s) |
| Geopolitical | ▲ base up (USD/JPY ↑) | 0.70 | 5% | Geopolitical tensions and a resilient US economy are likely to keep the USD strong against the JPY, despite Japan's efforts to stabilize its currency. | 3 doc(s) |
| Technical / Positioning | ▲ base up (USD/JPY ↑) | 0.80 | 10% | The technical indicators suggest a bullish trend for USD/JPY, supported by strong momentum and a favorable US economic backdrop, despite some recent pullbacks. | 5 doc(s) |

_Lean is expressed for the base currency: "base up" means USD strengthens against JPY (i.e. USD/JPY rises)._

## Risk Flags

- **Intervention risk** (Bank of Japan): Despite intervention risks, the persistent weakness of the JPY and the supportive US economic data suggest that the USD will appreciate against the JPY in the near term.
- **Geopolitical tail risk**: Geopolitical tensions and a resilient US economy are likely to keep the USD strong against the JPY, despite Japan's efforts to stabilize its currency.
- **Event risk (releases inside the horizon):**
| Date (UTC) | Country | Impact | Event | Consensus |
|---|---|---|---|---|
| 2026-07-27 12:30 | US | MEDIUM | Durable Goods Orders (Jun) | 1.6% |
| 2026-07-27 12:30 | US | MEDIUM | Durable Goods Orders ex Defense (Jun) | — |
| 2026-07-27 12:30 | US | MEDIUM | Durable Goods Orders ex Transportation (Jun) | 0.9% |
| 2026-07-27 12:30 | US | MEDIUM | Nondefense Capital Goods Orders ex Aircraft (Jun) | — |
| 2026-07-28 12:15 | US | MEDIUM | ADP Employment Change 4-week average (Jul) | — |
| 2026-07-28 13:00 | US | MEDIUM | Housing Price Index (MoM) (May) | — |
| 2026-07-28 14:00 | US | MEDIUM | Consumer Confidence (Jul) | — |

## Appendix — Sources

1. [Kevin Warsh Said the Fed Has "No Tolerance" for Inflation and Is Charting a "New Course" on Monetary Policy. J.P. Morgan Analysts Expect Rates to Hold Steady Through 2026.](https://app.bigdata.com/documents/C3AB1BE1FFB4EAA386990ED773879753?cnum=3) — Nasdaq (2026-07-24)
2. [Japanese Yen stays under pressure as resilient US economy supports the Dollar](https://app.bigdata.com/documents/03C6275257BE96F118519B4C349FD8AF?cnum=1) — FXStreet News (2026-07-24)
3. [Japanese Yen: Multi-decade lows against US Dollar face BoJ risk - Scotiabank](https://app.bigdata.com/documents/04781C401187F92D43FAB265BA6BA7E0?cnum=1) — FXStreet News (2026-07-24)
4. [New trade disruption overshadows strong production in Japan](https://app.bigdata.com/documents/96F442107B589742065F0CFD97680CC2?cnum=3) — The Economist (2026-07-24)
5. [Weekender - Friday, July 24, 2026](https://app.bigdata.com/documents/14B35752D3DBBF2598A97C8CFD965AA7?cnum=6) — Beta Securities S.A. (2026-07-24)
6. [Euro slips against US Dollar as strong US PMI supports Greenback](https://app.bigdata.com/documents/1B16BBAB7E6030763FDA848AF7D357CE?cnum=4) — FXStreet News (2026-07-24)
7. [[CIO View: What Matters Now] Bond Market Leads](https://app.bigdata.com/documents/46A0A505CEF83A88D5EF67FD5F2E6C1C?cnum=7) — Sohu (2026-07-24)
8. [Dollar could stay strong until US.-Iran ceasefire agreed](https://app.bigdata.com/documents/6FA6646E4B432DF029886A368A54E286?cnum=10) — MSN (2026-07-24)
9. [The Return of Tariffs](https://app.bigdata.com/documents/7DC015702649D8192CA7E70AB6448779?cnum=14) — Natixis (2026-07-24)
10. [Middle East Escalation Boosts Commodities As Monetary Policy Expectations Weigh On Currencies](https://app.bigdata.com/documents/E9A91588F018BD4D746DCA5B2D2D6090?cnum=6) — Blominvest Bank Sal (2026-07-24)
11. [Rates weekly: The point of no return?](https://app.bigdata.com/documents/ABE500695253435162FC20E4BB4A347E?cnum=46) — Natixis (2026-07-24)
12. [Orchid Island Capital, Inc.: Q2 2026 Earnings Call](https://app.bigdata.com/documents/EB9CC75708D8E81DF051A2F1D47D112F?cnum=13) — Factset Transcripts (2026-07-24)
13. [Bank of Japan set to hold rates at 1% as inflation expectations rise - Nikkei](https://app.bigdata.com/documents/60B1006FF5D1B1CAD29B88D4528095B2?cnum=1) — Yahoo! Finance (2026-07-24)
14. [Bank of Japan set to hold rates at 1% as inflation expectations rise - Nikkei](https://app.bigdata.com/documents/E5C878630A746E3D65E177C9298148A6?cnum=2) — Yahoo! Finance (2026-07-24)
15. [Michael Faulkender explains why the Fed should not raise rates due to temporary energy shocks](https://app.bigdata.com/documents/5EAB5CD40A9972ACD5055671FB2965D7?cnum=1) — FOX Business (2026-07-24)
16. [Japanese Yen: BoJ repricing may limit JPY losses against US Dollar - BBH](https://app.bigdata.com/documents/F42754B88FA2DDD884A56F92A4F8415B?cnum=1) — FXStreet News (2026-07-24)
17. [Euro pauses near 12-week high against Yen despite upbeat Eurozone PMI](https://app.bigdata.com/documents/F72E91F1C6CB9522389692E9F14D4BB9?cnum=3&cnum=4) — FXStreet News (2026-07-24)
18. [WEEK AHEAD: Fed, BoE and BoJ set to hold as Cook takes final Apple bow](https://app.bigdata.com/documents/4A392A7D9085A8375A794FC8D6A0FF8E?cnum=31) — Alliance News (2026-07-24)
19. [USD/JPY Breaks Records: Nothing Slows the Yen's Decline](https://app.bigdata.com/documents/FF7FA1D3D78C6876ED0ED2AAC3651B96?cnum=2) — Invest Macro (2026-07-24)
20. [Japan's $1.8 trillion pension giant might bring money home. That could jolt U.S. stocks and the Fed.](https://app.bigdata.com/documents/AD237D82336FF8A9ED5B56CDBE0996DC?cnum=3) — Yahoo! Finance (2026-07-24)
21. [Asia week ahead: Fed pause buys Asia breathing room ](https://app.bigdata.com/documents/055B3D85F1A3F08F67235051CBA1141B?cnum=1) — The Economist (2026-07-24)
22. [The specter of stagflation returns to haunt the global economy](https://app.bigdata.com/documents/4727F301A010A7E451CC4819FD34D9BE?cnum=3) — The Middle East: International Edition (2026-07-24)
23. [Federal Reserve: Holding rates while watching inflation risks - Commerzbank](https://app.bigdata.com/documents/D347621984F4DE3EE3D2E352D2327C13?cnum=1) — FXStreet News (2026-07-24)
24. [Here's Why The Fed Is Saying Less](https://app.bigdata.com/documents/0CB002689D40C4BAD2AE1B49B2CEBBFF?cnum=6) — Here's Why (2026-07-24)
25. [Japanese Yen remains pinned near 40-year low as Fed-BoJ rate gap keeps carry trade active](https://app.bigdata.com/documents/E97CEBD207C30C1948F57B84316B60F2?cnum=1) — FXStreet News (2026-07-24)
26. [Currency Option Volatility USD/JPY 1-week rises to around 7%](https://app.bigdata.com/documents/18413E4B2CE9EB99CACE5D6B19E9E8BB?cnum=1) — Livedoor (2026-07-24)
27. [Japan Flash PMI Hits Five-Month High as Factory Output Surges, Services Soften](https://app.bigdata.com/documents/FDBEA95E84FCDE7327491D4C97502B76?cnum=2) — MT Newswires - Asia Pacific (2026-07-24)
28. [International Financial Market Focus - Escalation of Iran Conflict, Concerns over High Corporate AI Capital Expenditure, Major US Stock Indices All Closed Lower](https://app.bigdata.com/documents/3A2CF52F305E0A18220333084764F43C?cnum=16) — Yuanta Financial Holding Co., Limited (2026-07-24)
29. [Weak yen and higher energy costs will prolong 2026 deficit](https://app.bigdata.com/documents/27BF95E1657CD43D43C3B9B7F99153DB?cnum=2) — The Economist (2026-07-23)
30. [Japanese Yen: Energy shock drives weaker currency - MUFG](https://app.bigdata.com/documents/98852DD4195AEB52CFC58C0D9FE7994C?cnum=1) — FXStreet News (2026-07-23)
31. [Economic Data Analysis - Japan's June Imports & Exports: AI Exports Remain Strong, but High Oil Prices and Weak Yen Widen Deficit](https://app.bigdata.com/documents/BE0B0799BB70BA87C15C03E3C27F21AD?cnum=3) — Yuanta Financial Holding Co., Limited (2026-07-23)
32. [Tokyo vows to take 'bold' action as yen keeps sliding](https://app.bigdata.com/documents/6E49B5B135F36F3B64B90D1A06BAA573?cnum=1) — Financial Times (2026-07-22)

---
*Data: Bigdata.com (country tearsheets + news/research search via the remote MCP server). Synthesis: OpenAI. This report is a research aid, not investment advice.*


## Generalizing to another pair

Re-run the notebook after editing only the parameters cell (Step 2). Swapping `base_country` / `quote_country` / central-bank names reruns the whole pipeline for any pair — `EUR/USD`, `GBP/JPY`, `USD/BRL`, `USD/TWD`, and so on.

For a country outside the tearsheet's supported set (e.g. Taiwan, `TW`), that side degrades gracefully to `bigdata_search` only; the run still completes and the report notes the reduced structured coverage. Adjust the per-pair weights in `config/drivers.py` (`PAIR_WEIGHT_OVERRIDES`) to reflect what drives each currency — e.g. rate differential + trade flows for an export-driven currency, risk sentiment / carry for a higher-beta EM currency.